## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

## 2. Define Paths

In [ ]:
BASE_PATH = "/content/drive/MyDrive/job-posting-classifier"

DATA_PROCESSED = BASE_PATH + "/data/processed/cleaned_data.csv"

BERT_MODEL_PATH = BASE_PATH + "/models/distilbert_model"

## 3. Install Libraries

In [ ]:
!pip install transformers datasets torch scikit-learn seaborn matplotlib

## 4. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from datasets import Dataset

import seaborn as sns
import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

## 5. Load Clean Dataset

In [ ]:
df = pd.read_csv(DATA_PROCESSED)

print("Dataset shape:", df.shape)
df.head()

## 6. Train/Test Split

In [ ]:
X = df["text"]
y = df["formatted_work_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 7. Convert Labels to Numbers

Transformers require numeric labels

In [ ]:
classes = sorted(df["formatted_work_type"].unique())

class_to_idx = {c:i for i,c in enumerate(classes)}
idx_to_class = {i:c for c,i in class_to_idx.items()}

y_train_idx = y_train.map(class_to_idx)
y_test_idx = y_test.map(class_to_idx)

print(class_to_idx)

## Class Weight Calculation

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_idx),
    y=y_train_idx
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

# Boost difficult classes
class_weights[class_to_idx["Contract"]] *= 1.5
class_weights[class_to_idx["Part-time"]] *= 2.5

class_weights = class_weights.to(device)

print("Class weights:", class_weights)

### Create Custom Trainer

In [ ]:
# import torch.nn as nn
# from transformers import Trainer

# class WeightedTrainer(Trainer):

#     def compute_loss(self, model, inputs, return_outputs=False, **kwargs):

#         labels = inputs.get("labels")

#         outputs = model(**inputs)

#         logits = outputs.get("logits")

#         loss_fct = nn.CrossEntropyLoss(weight=class_weights)

#         loss = loss_fct(
#             logits.view(-1, self.model.config.num_labels),
#             labels.view(-1)
#         )

#         return (loss, outputs) if return_outputs else loss

## 8. Create HuggingFace Dataset

In [ ]:
train_dataset = Dataset.from_dict({
    "text": X_train.tolist(),
    "label": y_train_idx.tolist()
})

test_dataset = Dataset.from_dict({
    "text": X_test.tolist(),
    "label": y_test_idx.tolist()
})

## 9. Load DistilBERT Tokenizer

In [ ]:
#model_name = "distilbert-base-uncased"
model_name = "microsoft/deberta-v3-small"

#tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

## 10. Tokenize Dataset

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=384
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

## 11. Load Model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(classes)
)

model = model.float().to(device)

## 12. Training Configuration

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    load_best_model_at_end=True,
    fp16=True,
    logging_steps=100
)

## Custom Trainer

In [ ]:
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

## 13. Evaluation Function

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds)
    }

## 14. Create Trainer

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

## 15. Train Model

In [ ]:
trainer.train()

## 16. Evaluate Model

In [ ]:
preds = trainer.predict(test_dataset)

y_pred = np.argmax(preds.predictions, axis=1)

print("Accuracy:", accuracy_score(y_test_idx, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test_idx, y_pred, target_names=classes))

In [ ]:
cm = confusion_matrix(y_test_idx, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("BERT Confusion Matrix")

plt.show()

## 17. Save Model

In [ ]:
model.save_pretrained(BERT_MODEL_PATH)
tokenizer.save_pretrained(BERT_MODEL_PATH)

print("BERT model saved at:", BERT_MODEL_PATH)